# F5 · A quantum self-attention layer for molecular generation
**Series F: Quantum + Machine Learning, Post 5** &nbsp;|&nbsp; companion *primer* notebook

Smaldone, Shee, Kyro, Farag, Chandani, Kyoseva and Batista (*J. Chem. Theory Comput.* 2025, 21, 5143) replace the query-key dot products of a transformer decoder by **quantum circuits**. Each attention score is the expectation value of a *modified Hadamard test* on $\log_2 d=6$ qubits; the value matrix and the rest of the decoder stay classical. The model is trained on QM9 SMILES with the **simultaneous perturbation stochastic approximation** (SPSA) and compared with classical decoders; the paper reports that the quantum and classical models perform alike.

This primer rebuilds, **in NumPy and without a quantum computer or simulator package**, what can be checked about that claim: a state-vector simulation of the circuits of the paper (Figures 3, 4 and 6); the **exact identity** between the measured ancilla expectation and the inner product $\mathrm{Re}\langle q_i|k_j\rangle$; what the layer can and cannot express (the scores are a bilinear form with an *orthogonal* matrix); the **cost accounting** (circuits per sequence, runtimes of Table 1, shot noise); the **gradient rules** (parameter shift and SPSA, and a surprise about the two-term rule on this circuit); a **reconstruction of the parameter counts** of the paper from the architecture; a consistency check of Table 2; and a **miniature hybrid transformer** trained with SPSA against classical and attention-free baselines. Part II runs the same ansatz in CUDA-Q and counts the parameters of the real models of the repository `anthonysmaldone/Quantum-Transformer`.

> **Two parts.** **Part I** needs only NumPy, pandas and matplotlib and takes about five minutes on a CPU (most of it is the training of four small models). **Part II** needs the repository, PyTorch and CUDA-Q (installed by the cells if possible); each stage is guarded.
>
> **Reading the code.** Every section starts with a short **Stop and predict** question: answer it before running the cell.

| Section | What you do |
|---|---|
| 1 | the classical score and what the circuit has to produce |
| 2 | gates, registers and the ansatz of Figure 3 |
| 3 | the modified Hadamard test of Figures 4 and 6, simulated exactly |
| 4 | what the layer can express: an orthogonal bilinear form |
| 5 | circuits, shots and runtimes |
| 6 | gradients: parameter shift, SPSA |
| 7 | the parameter counts and Table 2, rebuilt |
| 8 | a miniature hybrid transformer against classical baselines |
| **Part II** | CUDA-Q and the repository |

## 0. Setup

In [ ]:
import math, time, itertools, importlib.util, os, sys, re, glob
import numpy as np, pandas as pd, matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True, linewidth=120)
plt.rcParams.update({"figure.dpi": 110, "font.size": 10, "axes.spines.top": False, "axes.spines.right": False})
COLORS = dict(blue="#0072B2", orange="#E69F00", green="#009E73", red="#D55E00", pink="#CC79A7", gray="#7f7f7f")
rng = np.random.default_rng(0)

## 1. What the circuit has to produce
For a sequence of $n$ tokens with embeddings $z_i\in\mathbb R^d$, classical self-attention forms $Q=ZW^Q$, $K=ZW^K$ and the scores $a_{ij}=q_i\cdot k_j/\sqrt{d_k}$, followed by a softmax over $j\le i$ (causal mask) and a product with the values $V=ZW^V$. The cost of $QK^T$ is $O(n^2d)$. The hybrid layer replaces $a_{ij}$ by the expectation value $\mathrm{Re}\langle q_i|k_j\rangle$ of unit vectors $|q_i\rangle,|k_j\rangle$ in a space of dimension $d=2^m$ with $m$ qubits (here $m=6$, $d=64$), *scaled by* $\sqrt{d_k}$. A dot product of unit vectors lies in $[-1,1]$; the paper multiplies it by $\sqrt{d_k}$ so that the scores have a variance of about $1$.
**Stop and predict:** for random unit vectors in $\mathbb R^{64}$, what is the standard deviation of the dot product, and of the dot product times $\sqrt{64}$?

In [ ]:
d = 64; u = rng.normal(size=(20000, d)); v = rng.normal(size=(20000, d)); u /= np.linalg.norm(u, axis=1, keepdims=True); v /= np.linalg.norm(v, axis=1, keepdims=True)
dots = (u * v).sum(1); print(f"random unit vectors in R^{d}: std of the dot product {dots.std():.4f} (1/sqrt(d) = {1 / math.sqrt(d):.4f}); times sqrt(d): {math.sqrt(d) * dots.std():.3f}")
x = rng.normal(size=(20000, d)); y = rng.normal(size=(20000, d)); print(f"classical scores of unit-variance vectors, (q.k)/sqrt(d): std {((x * y).sum(1) / math.sqrt(d)).std():.3f}")
assert abs(math.sqrt(d) * dots.std() - 1) < 0.03

## 2. Gates, registers and the ansatz of Figure 3
Every circuit of the paper uses one layer of $R_y(\theta)=\begin{pmatrix}\cos\theta/2&-\sin\theta/2\\ \sin\theta/2&\cos\theta/2\end{pmatrix}$ gates followed by a layer of CNOT gates. Figure 3 shows three qubits; we continue the pattern as a **ring** of CNOTs $0\to1\to\dots\to m-1\to0$ for registers of other sizes (this reading of the figure is ours; every identity below holds for any real ansatz). A *token register* and a *position register* of $3$ qubits each are prepared from $|000\rangle$ by $U_e(\theta_{e_i})$ and $U_p(\theta_{p_i})$, giving $|z_i\rangle=|e_i\rangle\otimes|p_i\rangle$ on $6$ qubits; the query and key unitaries $U_q,U_k$ are the same ansatz on all $6$ qubits with $6$ angles each.
**Stop and predict:** the gates are all real matrices. What does that imply for $|q_i\rangle$, $|k_j\rangle$ and $\langle q_i|k_j\rangle$?

In [ ]:
def ry(t): c, s = math.cos(t / 2), math.sin(t / 2); return np.array([[c, -s], [s, c]])
def cnot(n, c, t):
    M = np.zeros((2 ** n, 2 ** n))
    for b in range(2 ** n):
        bits = [(b >> (n - 1 - k)) & 1 for k in range(n)]
        if bits[c]: bits[t] ^= 1
        M[sum(v << (n - 1 - k) for k, v in enumerate(bits)), b] = 1
    return M
def ansatz(theta):
    """one layer of Ry gates, then a ring of CNOTs (qubit 0 is the most significant)"""
    n = len(theta); U = np.array([[1.0]])
    for t in theta: U = np.kron(U, ry(t))
    for k in range(n): U = cnot(n, k, (k + 1) % n) @ U
    return U
def zstate(te, pe, ce=None):
    z0 = np.zeros(8); z0[0] = 1; s = np.kron(ansatz(te) @ z0, ansatz(pe) @ z0); return s
th = rng.uniform(0, np.pi, 6); U = ansatz(th)
print("U is orthogonal:", np.allclose(U.T @ U, np.eye(64)), "| entries real:", np.isrealobj(U), "| the state |z> has norm", np.linalg.norm(zstate(*rng.uniform(0, np.pi, (2, 3)))))
print("so |q> = U_q|z> and |k> = U_k|z'> are real unit vectors, and <q|k> is a real dot product (the 'Re' of the paper is not needed)")
assert np.allclose(U.T @ U, np.eye(64))

## 3. The modified Hadamard test of Figures 4 and 6, simulated exactly
The circuit of Figure 4 uses an ancilla and the $6$ working qubits ($7$ qubits, $128$ amplitudes). Following Equations 7--14: Hadamard on the ancilla; prepare $|z_i\rangle$; apply $U_q$; a **controlled** $U_q^\dagger$, which maps the ancilla-1 branch back to $|z_i\rangle$; controlled inverses of the embedding unitaries, which reset that branch to $|0\ldots0\rangle$; controlled preparation of $|z_j\rangle$; controlled $U_k$; Hadamard on the ancilla; measure $Z$. The state before the measurement is $(|0\rangle(|q_i\rangle+|k_j\rangle)+|1\rangle(|q_i\rangle-|k_j\rangle))/2$, so $\langle Z\rangle=\mathrm{Re}\langle q_i|k_j\rangle$.
**Stop and predict:** how closely will $\langle Z\rangle$ from the $128$-amplitude simulation match the dot product of the two $64$-vectors?

In [ ]:
H1 = np.array([[1, 1], [1, -1]]) / math.sqrt(2); I2 = np.eye(2)
def ctrl(U): n = U.shape[0]; return np.block([[np.eye(n), np.zeros((n, n))], [np.zeros((n, n)), U]])
def prep_unitary(te, pe): return np.kron(ansatz(te), np.eye(8)) @ np.kron(np.eye(8), ansatz(pe))          # |000>|000> -> |e>|p>
def hadamard_test(te_i, pe_i, te_j, pe_j, tq, tk):
    Zi, Zj, Uq, Uk = prep_unitary(te_i, pe_i), prep_unitary(te_j, pe_j), ansatz(tq), ansatz(tk)
    psi = np.kron(H1 @ np.array([1.0, 0]), np.eye(64)[0])                  # ancilla in |+>, registers in |0...0>
    psi = np.kron(I2, Zi) @ psi; psi = np.kron(I2, Uq) @ psi              # Psi_2 = |+>|q_i>
    psi = ctrl(Uq.T) @ psi                                                 # Psi_3 = (|0>|q_i> + |1>|z_i>)/sqrt2
    psi = ctrl(Zi.T) @ psi                                                 # Psi_4: the ancilla-1 branch is reset to |0...0>
    psi = ctrl(Zj) @ psi; psi = ctrl(Uk) @ psi                             # Psi_5, Psi_6 = (|0>|q_i> + |1>|k_j>)/sqrt2
    psi = np.kron(H1, np.eye(64)) @ psi                                    # Psi_7
    return psi @ np.kron(np.diag([1.0, -1]), np.eye(64)) @ psi, psi
errs = []
for _ in range(100):
    a = rng.uniform(0, np.pi, (4, 3)); tq_, tk_ = rng.uniform(0, np.pi, (2, 6)); e, _ = hadamard_test(*a, tq_, tk_)
    errs.append(abs(e - (ansatz(tq_) @ zstate(a[0], a[1])) @ (ansatz(tk_) @ zstate(a[2], a[3]))))
print(f"100 random circuits: largest |<Z> - <q|k>| = {max(errs):.1e}")
e, psi = hadamard_test(*rng.uniform(0, np.pi, (4, 3)), *rng.uniform(0, np.pi, (2, 6))); p0 = (psi.reshape(2, 64) ** 2).sum(1)[0]; print(f"probability of the outcome 0 on the ancilla: {p0:.4f} = (1 + <q|k>)/2 = {(1 + e) / 2:.4f}")
assert max(errs) < 1e-12 and abs(p0 - (1 + e) / 2) < 1e-12

In [ ]:
# the conditional circuit of Figure 6: three registers of 2 qubits (token, position, physicochemical); the property register c is the same for i and j and is not reset
def reg_ops(te, pe): return np.kron(ansatz(te), np.eye(16)) @ np.kron(np.kron(np.eye(4), ansatz(pe)), np.eye(4))          # token and position registers of 2 qubits each, inside 6 qubits
def prep3(te, pe, ce): return reg_ops(te, pe) @ np.kron(np.eye(16), ansatz(ce))
def hadamard_test_cond(te_i, pe_i, te_j, pe_j, ce, tq, tk):
    Uq, Uk = ansatz(tq), ansatz(tk); Zi, Zj = prep3(te_i, pe_i, ce), prep3(te_j, pe_j, ce)
    Ei, Ej = reg_ops(te_i, pe_i), reg_ops(te_j, pe_j)
    psi = np.kron(H1 @ np.array([1.0, 0]), np.eye(64)[0]); psi = np.kron(I2, Zi) @ psi; psi = np.kron(I2, Uq) @ psi; psi = ctrl(Uq.T) @ psi
    psi = ctrl(Ei.T) @ psi                         # only the token and position registers are reset; |c> stays
    psi = ctrl(Ej) @ psi; psi = ctrl(Uk) @ psi; psi = np.kron(H1, np.eye(64)) @ psi
    z0 = np.zeros(4); z0[0] = 1; zi = np.kron(np.kron(ansatz(te_i) @ z0, ansatz(pe_i) @ z0), ansatz(ce) @ z0); zj = np.kron(np.kron(ansatz(te_j) @ z0, ansatz(pe_j) @ z0), ansatz(ce) @ z0)
    return psi @ np.kron(np.diag([1.0, -1]), np.eye(64)) @ psi, (Uq @ zi) @ (Uk @ zj)
errs = [abs(np.subtract(*hadamard_test_cond(*rng.uniform(0, np.pi, (5, 2)), *rng.uniform(0, np.pi, (2, 6))))) for _ in range(50)]
print(f"conditional circuit with a shared property register, 50 random cases: largest difference {max(errs):.1e}"); assert max(errs) < 1e-12

## 4. What the layer can express
The simulation shows what the circuit computes: $a_{ij}=\langle q_i|k_j\rangle=z_i^TU_q^TU_kz_j$, with real orthogonal $U_q,U_k$. Two consequences. **(i)** Only the orthogonal matrix $O=U_q^TU_k$ enters: $(U_q,U_k)$ and $(I,O)$ give the same scores. For the single-layer ansatz $U=E\,\bigotimes_mR_y(\theta_m)$ with a fixed entangling layer $E$ (a permutation matrix), $O=\bigotimes_mR_y(-\theta^q_m)\,E^TE\,\bigotimes_mR_y(\theta^k_m)=\bigotimes_mR_y(\theta^k_m-\theta^q_m)$: **the entangling layer cancels and only the six differences $\theta^k_m-\theta^q_m$ matter**, so the $12$ query and key angles carry six effective parameters. **(ii)** The embeddings are *product states* $|e_i\rangle\otimes|p_i\rangle$ of two $8$-dimensional unit vectors, so the scores are a bilinear form with an orthogonal matrix between product states of token and position. Compare this with the classical layer, where $W^Q$ and $W^K$ are arbitrary $64\times64$ matrices.
**Stop and predict:** how many of the $12$ query and key angles change the scores independently, and does the ring of CNOTs inside $U_q$ and $U_k$ matter?

In [ ]:
tq, tk = rng.uniform(0, np.pi, (2, 6)); a = rng.uniform(0, np.pi, (4, 3)); zi, zj = zstate(a[0], a[1]), zstate(a[2], a[3])
Uq, Uk = ansatz(tq), ansatz(tk); O = Uq.T @ Uk
print(f"score with (Uq, Uk): {(Uq @ zi) @ (Uk @ zj):.12f} | with (I, O = Uq^T Uk): {zi @ O @ zj:.12f} | O orthogonal: {np.allclose(O.T @ O, np.eye(64))}")
Oprod = np.array([[1.0]])
for a_, b_ in zip(tq, tk): Oprod = np.kron(Oprod, ry(b_ - a_))
print("O equals the tensor product of the six rotations Ry(theta_k - theta_q), with no trace of the CNOTs:", np.allclose(O, Oprod), "| score from the six differences alone:", f"{zi @ Oprod @ zj:.12f}")
# the Jacobian of O with respect to the 12 angles has rank <= 12 inside the 2016-dimensional group SO(64)
def flatO(t): return (ansatz(t[:6]).T @ ansatz(t[6:])).ravel()
t0 = np.concatenate([tq, tk]); J = np.stack([(flatO(t0 + 1e-6 * e) - flatO(t0 - 1e-6 * e)) / 2e-6 for e in np.eye(12)], 1); print("rank of the Jacobian of O with respect to the 12 angles:", np.linalg.matrix_rank(J, tol=1e-7), "(six directions, theta_q + c and theta_k + c for a common shift c, leave O unchanged)")
# the variance of the scaled score over random angles, as claimed in the paper
sc = np.array([8 * (ansatz(r[:6]) @ zstate(r[6:9], r[9:12])) @ (ansatz(r[12:]) @ zstate(r[6:9] + 0, r[9:12] + 0)) for r in rng.uniform(0, np.pi, (2000, 18))])
sc2 = np.array([8 * (ansatz(r[:6]) @ zstate(r[6:9], r[9:12])) @ (ansatz(r[12:18]) @ zstate(r[18:21], r[21:24])) for r in rng.uniform(0, np.pi, (2000, 24))])
print(f"scaled scores 8<q|k> over random angles: mean {sc2.mean():.3f}, std {sc2.std():.3f}, range [{sc2.min():.2f}, {sc2.max():.2f}] (classical scaled scores have std 1)")
assert abs((Uq @ zi) @ (Uk @ zj) - zi @ O @ zj) < 1e-13 and np.allclose(O, Oprod) and np.linalg.matrix_rank(J, tol=1e-7) == 6

## 5. Circuits, shots and runtimes
For a sequence of $n$ tokens the causal attention matrix needs $n(n+1)/2$ scores; the first, $a_{1,1}$, has a softmax of $1$ and is skipped: $n(n+1)/2-1$ circuits. Circuits with identical parameters give identical results and are computed once. Table 1 of the paper gives the time of one epoch on QM9 for a batch of $256$.
**Stop and predict:** with a block of $24$ positions and a batch of $256$ sequences, how many circuits is that per batch, and how many of the training epochs of the paper fit in one of the $48$-hour jobs of the repository's example script?

In [ ]:
n = 24; per_seq = n * (n + 1) // 2 - 1; n_train = round(133798 * 20 / 21); batches = -(-n_train // 256)
print(f"circuits per sequence {per_seq}; per batch of 256: {256 * per_seq:,}; batches per epoch {batches}; circuits per epoch (forward only): {batches * 256 * per_seq:,}")
tab = pd.DataFrame({"hardware": ["CPU", "1 GPU", "4 GPUs"], "epoch (h), Table 1": [41.28, 30.85, 8.03]}); tab["seconds per batch update"] = tab["epoch (h), Table 1"] * 3600 / batches; tab["20 epochs (h)"] = tab["epoch (h), Table 1"] * 20; tab["20 epochs (days)"] = tab["20 epochs (h)"] / 24
print(tab.round(2).to_string(index=False)); print(f"speedups: CPU/1 GPU {41.28 / 30.85:.3f} (paper 1.34), 1 GPU/4 GPUs {30.85 / 8.03:.3f} (paper 3.84), CPU/4 GPUs {41.28 / 8.03:.2f}; 20 epochs on 4 GPUs = {20 * 8.03 / 48:.1f} jobs of 48 hours")
# redundancy: unique parameter tuples (token_i, position_i, token_j, position_j) among the causal pairs of a batch of 256 toy sequences
import itertools as it
toy = [[int(t) for t in rng.integers(1, 12, 24)] for _ in range(256)]; tuples = {(s[i], i, s[j], j) for s in toy for i in range(24) for j in range(i + 1)}; print(f"a batch of 256 random sequences over 11 tokens: {256 * 24 * 25 // 2:,} pairs, {len(tuples):,} distinct (token_i, pos_i, token_j, pos_j) tuples")
assert per_seq == 299 and batches == 498 and abs(41.28 / 30.85 - 1.34) < 0.005

In [ ]:
# shot noise: on hardware <Z> is estimated from N shots; the estimate has a standard deviation sqrt((1 - r^2)/N), and Hoeffding gives N = 2 ln(2/delta) / eps^2 for |error| <= eps
def hoeffding_shots(eps, delta=0.05): return math.ceil(2 * math.log(2 / delta) / eps ** 2)
print("shots for an error eps in <Z> with probability 0.95:", {e: f"{hoeffding_shots(e):,}" for e in (0.1, 0.03, 0.01)}, "| after the factor 8 on the scores, a logit error is 8 eps")
rows = [np.array([(ansatz(r[:6]) @ zstate(r[6 + 6 * k:9 + 6 * k], r[9 + 6 * k:12 + 6 * k])) @ (ansatz(r[:6]) @ zstate(r[6:9], r[9:12])) for k in range(10)]) for r in rng.uniform(0, np.pi, (300, 66))]
def tv_error(N):
    out = []
    for r in rows:
        r = np.clip(r, -1, 1); est = 2 * rng.binomial(N, (1 + r) / 2) / N - 1; p = np.exp(8 * r - (8 * r).max()); p /= p.sum(); q = np.exp(8 * est - (8 * est).max()); q /= q.sum(); out.append(0.5 * np.abs(p - q).sum())
    return np.mean(out)
Ns = [30, 100, 300, 1000, 3000, 10000, 100000]; tv = [tv_error(N) for N in Ns]; slope = np.polyfit(np.log(Ns), np.log(tv), 1)[0]
print(pd.DataFrame({"shots per score": Ns, "mean total-variation error of an attention row": np.round(tv, 4)}).to_string(index=False)); print(f"log-log slope {slope:.2f} (shot noise: -0.5)")
assert -0.62 < slope < -0.4

## 6. Gradients: parameter shift and SPSA
The paper (Equation 20) writes the **parameter-shift rule** $\partial f/\partial\theta=\tfrac12[f(\theta+\pi/2)-f(\theta-\pi/2)]$, exact for a rotation $e^{-i\theta P/2}$ in an *expectation value of an observable*, at two circuit evaluations per parameter. Here the measured quantity is an *interference amplitude*, $f=z_i^TU_q^T(\theta)U_kz_j$, which is linear in $\cos(\theta/2)$ and $\sin(\theta/2)$ for each angle (the expectation of an observable would be linear in $\cos\theta,\sin\theta$). We test the rule on this circuit, together with the alternative $f'(\theta)=\tfrac12f(\theta+\pi)$ that follows from the half-angle form. Then **SPSA** (Equations 21--22): perturb all $m$ parameters at once by $\pm\epsilon$ along a random sign vector $\Delta$, and use $[f(x+\epsilon\Delta)-f(x-\epsilon\Delta)]/(2\epsilon)\cdot\Delta^{-1}$ as a gradient estimate from two evaluations.
**Stop and predict:** by what factor does the two-term $\pi/2$ rule miss the derivative of this score?

In [ ]:
def score(x):   # x = 24 angles: token_i, pos_i, token_j, pos_j (3 each), U_q (6), U_k (6); the exact expectation of the Hadamard test
    return (ansatz(x[12:18]) @ zstate(x[0:3], x[3:6])) @ (ansatz(x[18:24]) @ zstate(x[6:9], x[9:12]))
x0 = rng.uniform(0, np.pi, 24); eps = 1e-6
exact = np.array([(score(x0 + eps * e) - score(x0 - eps * e)) / (2 * eps) for e in np.eye(24)])
two_term = np.array([(score(x0 + np.pi / 2 * e) - score(x0 - np.pi / 2 * e)) / 2 for e in np.eye(24)]); one_eval = np.array([score(x0 + np.pi * e) / 2 for e in np.eye(24)])
ok = np.abs(exact) > 1e-4
print(f"two-term rule at pi/2 divided by the exact derivative, over {ok.sum()} of the 24 angles: min {np.min(two_term[ok] / exact[ok]):.4f}, max {np.max(two_term[ok] / exact[ok]):.4f} (sqrt 2 = {math.sqrt(2):.4f})")
print(f"single-evaluation rule f(theta + pi)/2 against the exact derivative: largest difference {np.abs(one_eval - exact).max():.1e}")
print("so the two-term rule of Equation 20 gives sqrt(2) times the derivative of this score, and one evaluation at a shift of pi is exact; training in the paper uses SPSA, which does not depend on this rule")
assert np.allclose(two_term[ok] / exact[ok], math.sqrt(2), atol=1e-3) and np.abs(one_eval - exact).max() < 1e-6

In [ ]:
# SPSA: the quality of one estimate against the true gradient, as the number of parameters m grows (a smooth test function with equal-sized partial derivatives)
def spsa_cos(m, K=400, eps=0.01):
    a = rng.uniform(0.5, 1.5, m); ph = rng.uniform(0, 2 * np.pi, m); f = lambda x: np.sum(a * np.cos(x + ph)); x = rng.uniform(0, 2 * np.pi, m); g = -a * np.sin(x + ph); cs = []
    for _ in range(K):
        D = rng.choice([-1.0, 1.0], m); gh = (f(x + eps * D) - f(x - eps * D)) / (2 * eps) * D; cs.append(gh @ g / (np.linalg.norm(gh) * np.linalg.norm(g)))
    return np.mean(cs)
ms = [4, 16, 64, 183, 512, 2048]; cs = [spsa_cos(m) for m in ms]
print(pd.DataFrame({"m (parameters)": ms, "cos(SPSA estimate, true gradient)": np.round(cs, 3), "1/sqrt(m)": np.round(1 / np.sqrt(ms), 3)}).to_string(index=False))
# averaging K independent SPSA estimates recovers the gradient; the cost is 2K evaluations, the same as K parameter-shift-like sweeps would need for K parameters
m = 183; a = rng.uniform(0.5, 1.5, m); ph = rng.uniform(0, 2 * np.pi, m); f = lambda x: np.sum(a * np.cos(x + ph)); x = rng.uniform(0, 2 * np.pi, m); g = -a * np.sin(x + ph)
for K in (1, 10, 100, 1000):
    gh = np.mean([(f(x + 0.01 * D) - f(x - 0.01 * D)) / 0.02 * D for D in rng.choice([-1.0, 1.0], (K, m))], 0); print(f"m = 183, average of K = {K:4d} estimates: cosine with the true gradient {gh @ g / (np.linalg.norm(gh) * np.linalg.norm(g)):.3f}")
fig, ax = plt.subplots(1, 2, figsize=(10.5, 3.6)); ax[0].loglog(Ns, tv, "o-", color=COLORS["blue"]); ax[0].set(xlabel="shots per score", ylabel="error of an attention row (TV)", title="(a) shot noise, slope -1/2")
ax[1].loglog(ms, cs, "o-", color=COLORS["orange"], label="SPSA, one estimate"); ax[1].loglog(ms, 1 / np.sqrt(ms), "k--", label=r"$1/\sqrt{m}$"); ax[1].set(xlabel="number of parameters m", ylabel="cosine with the true gradient", title="(b) SPSA estimate quality"); ax[1].legend(fontsize=8); plt.tight_layout(); plt.show()
assert abs(cs[2] - 1 / math.sqrt(64)) < 0.04

## 7. The parameter counts and Table 2, rebuilt
The paper gives the numbers of parameters of its models: $47{,}704$ (hybrid, sequence-only), $55{,}713$ (classical with $64\times64$ query and key matrices) and, for the condition-based models, $48{,}307$ and $56{,}535$. If we know the architecture they should be reproducible. The decoder has one block with one head: token and position embeddings of width $64$ (used by the value path); the value matrix $W^V$ ($64\times64$), the output projection with bias, two layer normalizations, a feed-forward network $64\to256\to64$ with biases, a final layer normalization and a linear head with bias; the hybrid model adds a table of token angles ($3$ per token, $2$ in the condition-based model), a table of position angles, and $6+6$ angles for $U_q$ and $U_k$; the condition-based models add a linear map of the nine properties to $64$ dimensions and, for the hybrid, a second one to the $2$ angles of the property register.
**Stop and predict:** which vocabulary size and block size reproduce $47{,}704$ and $55{,}713$ exactly?

In [ ]:
def base(V, B, d=64):
    return V * d + B * d + d * d + (d * d + d) + 2 * (2 * d) + (d * 4 * d + 4 * d + 4 * d * d + d) + 2 * d + (d * V + V)
def counts(V=33, B=24, d=64):
    q_seq = base(V, B) + 3 * V + 3 * B + 12; c_seq = base(V, B) + 2 * d * d; phys, ang = 9 * d + d, 9 * 2 + 2
    return dict(q_seq=q_seq, c_seq=c_seq, q_cond=base(V, B) + 2 * V + 2 * B + 12 + phys + ang, c_cond=c_seq + phys)
paper = dict(q_seq=47704, c_seq=55713, q_cond=48307, c_cond=56535); got = counts()
print(pd.DataFrame({"model": ["hybrid, sequence only", "classical, sequence only", "hybrid, conditions", "classical, conditions"], "paper": list(paper.values()), "rebuilt (V=33, block 24)": list(got.values()), "difference": [paper[k] - got[k] for k in paper]}).to_string(index=False))
print("only V + B is fixed by the sequence-only counts: 3(V + B) + 12 = 2 * 64^2 - (55713 - 47704) ->", 3 * (33 + 24) + 12, "=", 2 * 64 * 64 - (55713 - 47704), "| the 33 tokens of the paper (30 + pad, start, end) give a block of 24 positions")
for V_, B_ in ((35, 22), (32, 25)): c = counts(V_, B_); print(f"  V = {V_}, block {B_}: hybrid {c['q_seq']:,}, classical {c['c_seq']:,}")
print("the classical condition-based model is 182 parameters above the rebuilt count: an accounting difference in how the properties enter the classical model")
assert got["q_seq"] == 47704 and got["c_seq"] == 55713 and got["q_cond"] == 48307 and paper["c_cond"] - got["c_cond"] == 182

In [ ]:
# Table 2: validity V, uniqueness U and their product V x U as reported (percent)
T2 = {("sequence only", "Quantum"): (68.6, 81.9, 56.2), ("sequence only", "Classical-eq"): (69.4, 79.4, 55.1), ("sequence only", "Classical"): (72.5, 81.2, 55.9),
      ("property embeddings", "Quantum"): (50.5, 38.8, 19.6), ("property embeddings", "Classical-eq"): (50.7, 40.0, 20.3), ("property embeddings", "Classical"): (38.5, 48.0, 18.5)}
rows = [(a, b, v, u, vu, round(v * u / 100, 2), round(vu - v * u / 100, 2)) for (a, b), (v, u, vu) in T2.items()]
print(pd.DataFrame(rows, columns=["setting", "model", "validity", "uniqueness", "V x U reported", "V x U from V and U", "reported - computed"]).to_string(index=False))
A1 = {"mean": [(50.5, 38.8, 19.6), (50.7, 40.0, 20.3), (38.5, 48.0, 18.5)], "median": [(70.2, 14.5, 10.2), (80.3, 14.5, 11.6), (70.9, 23.1, 16.4)], "mode": [(73.8, 21.7, 16.0), (65.8, 22.6, 14.9), (64.9, 28.0, 18.2)]}
print("Table A1 (nine rows): largest difference between the reported V x U and V*U:", max(abs(vu - v * u / 100) for r in A1.values() for v, u, vu in r))
bad = [(a, b) for (a, b), (v, u, vu) in T2.items() if abs(vu - v * u / 100) > 0.06]; print("rows of Table 2 that do not satisfy V x U:", bad)
assert bad == [("sequence only", "Classical")]

In [ ]:
# Table 3 of the paper (conditional generation): which model is closest to the target for each of the nine properties? Ties are shared
P9 = ["MW", "HBA", "HBD", "nRot", "nRing", "nHet", "TPSA", "logP", "Stereo"]; MODELS = ["Quantum", "Classical-eq", "Classical"]
T3 = {"mean": ([122.77, 2.23, 0.83, 0.92, 1.74, 2.47, 37.16, 0.30, 1.71], [[125.13, 2.31, 0.55, 0.63, 2.05, 2.37, 32.58, 0.30, 2.11], [123.42, 2.15, 0.55, 0.55, 2.03, 2.21, 31.74, 0.46, 1.98], [128.04, 2.15, 1.00, 0.79, 1.67, 2.19, 34.53, 0.52, 1.81]]),
      "mean + 2 sd": ([137.88, 4.34, 2.50, 3.10, 4.16, 4.84, 79.67, 2.30, 4.77], [[135.73, 3.95, 2.79, 2.71, 3.74, 4.82, 81.61, 2.24, 4.46], [137.12, 3.98, 2.75, 2.52, 3.62, 4.76, 78.03, 2.36, 4.34], [136.09, 4.05, 2.90, 2.91, 3.53, 4.78, 83.44, 2.37, 4.27]]),
      "mean - 2 sd": ([107.66, 0.12, 0.00, 0.00, 0.00, 0.10, 0.00, -1.71, 0.00], [[112.83, 0.63, 0.15, 0.11, 0.00, 0.05, 0.55, -1.74, 0.41], [112.12, 0.58, 0.19, 0.11, 0.00, 0.06, 0.67, -1.49, 0.26], [113.33, 0.30, 0.33, 0.11, 0.00, 0.06, 0.66, -1.46, 0.15]])}
tot = {m: 0.0 for m in MODELS}; rows = []
for sec, (tg, vals) in T3.items():
    err = np.abs(np.array(vals) - np.array(tg)); w = {m: 0.0 for m in MODELS}; ties = []
    for j, p_ in enumerate(P9):
        win = [MODELS[i] for i in range(3) if abs(err[i, j] - err[:, j].min()) < 1e-9]
        if len(win) > 1: ties.append(p_)
        for m in win: w[m] += 1 / len(win); tot[m] += 1 / len(win)
    rows.append((sec, *[round(w[m], 2) for m in MODELS], ", ".join(ties)))
print(pd.DataFrame(rows, columns=["target", *MODELS, "properties with a tie"]).to_string(index=False)); print("total over the 27 comparisons, ties shared:", {m: round(v, 1) for m, v in tot.items()})
print("for the mean row, hydrogen-bond acceptors are a three-way tie (each model is 0.08 away from 2.23), although the paper counts it for the quantum model")
assert abs(sum(tot.values()) - 27) < 1e-9 and abs(tot["Quantum"] - 9.5) < 0.01 and abs(tot["Classical"] - 10.5) < 0.01

## 8. A miniature hybrid transformer against classical baselines
We now train the hybrid layer the way the paper does: **exact expectation values** for the scores (our simulation gives them without shot noise), **SPSA** for all angles ($m=183$ in this vocabulary and block size: $3\cdot18+3\cdot39+12$) and an ordinary gradient (Adam) for the classical parameters. The task is a SMILES language model on a small corpus ($1{,}600$ molecules built from ten scaffolds, $1{,}400$ for training and $200$ held out), width $64$, one block, one head. We compare **four** attention layers: the hybrid one; a classical one with full $64\times64$ query and key matrices; a classical one with rank-$3$ queries and keys ($64\times3$, the size of the reduced model in the repository's documentation); and an **attention-free** baseline whose scores are all zero (uniform causal attention), a control that the paper does not report.
**Stop and predict:** where will the hybrid model end: with the classical layers, with the attention-free baseline, or between?

In [ ]:
SCAF = {"benzene": "c1c{A}cc{B}cc1", "pyridine": "n1cc{A}cc{B}c1", "cyclohexane": "C1CC{A}CC{B}C1", "piperidine": "N({B})1CCC{A}CC1", "naphthalene": "c1cc{A}c2ccccc2c1", "biphenyl": "c1cc{A}ccc1-c1ccc{B}cc1",
        "phenethylpiperidine": "c1cc{A}ccc1CCN1CCCC{B}C1", "diphenylmethane": "c1ccccc1Cc1ccc{A}cc1", "indole": "c1cc{A}c2[nH]ccc2c1", "cyclopentane": "C1CCC{A}C1"}
SUB = ["", "F", "Cl", "Br", "C", "CC", "CCC", "OC", "O", "N", "C(F)(F)F", "C#N", "CO", "CN", "OCC", "C(C)C"]
TOKEN = re.compile(r"Cl|Br|\[[^\]]+\]|[A-Za-z]|\d|[()=#\-+/\\@.]")
build = lambda name, a, b: SCAF[name].replace("{A}", f"({a})" if a else "").replace("{B}", f"({b})" if b else "")
seen, smi = set(), []
for name in SCAF:
    for a, b in itertools.product(SUB, SUB):
        s = build(name, a, b)
        if s not in seen: seen.add(s); smi.append(s)
itos = ["<pad>", "<cls>", "<eos>"] + sorted({t for s in smi for t in TOKEN.findall(s)}); stoi = {t: i for i, t in enumerate(itos)}; V = len(itos); T = max(len(TOKEN.findall(s)) for s in smi) + 1; D = 64
ids_all = np.array([[stoi[t] for t in TOKEN.findall(s)] + [2] + [0] * (T - len(TOKEN.findall(s)) - 1) for s in smi]); perm = np.random.default_rng(0).permutation(len(smi)); tr_i, te_i = perm[:1400], perm[1400:]
def softmax(x, axis=-1): x = x - x.max(axis, keepdims=True); e = np.exp(x); return e / e.sum(axis, keepdims=True)
# the quantum scores: angles -> product states -> orthogonal map -> 8 <q|k>, in closed form (this equals the Hadamard test of Section 3)
def ring_perm(n):
    P = np.zeros((2 ** n, 2 ** n))
    for b in range(2 ** n):
        bits = [(b >> (n - 1 - k)) & 1 for k in range(n)]
        for k in range(n):
            if bits[k]: bits[(k + 1) % n] ^= 1
        P[sum(v << (n - 1 - k) for k, v in enumerate(bits)), b] = 1
    return P
R3, R6 = ring_perm(3), ring_perm(6)
def reg_states(theta):
    c, s = np.cos(theta / 2), np.sin(theta / 2); a = np.stack([c, s], -1); prod = np.einsum("na,nb,nc->nabc", a[:, 0], a[:, 1], a[:, 2]).reshape(len(theta), 8); return prod @ R3.T
def ansatz6(th):
    U = np.array([[1.0]])
    for t in th: U = np.kron(U, ry(t))
    return R6 @ U
def q_scores(theta, ids):
    B_, T_ = ids.shape; et = reg_states(theta["tok"])[ids]; ep = np.repeat(reg_states(theta["pos"][:T_])[None], B_, 0)
    z = np.einsum("bta,btc->btac", et, ep).reshape(B_, T_, 64); return 8.0 * (z @ ansatz6(theta["q"]).T) @ (z @ ansatz6(theta["k"]).T).transpose(0, 2, 1)
KEYS = ["tok", "pos", "q", "k"]
def init_theta(seed): r = np.random.default_rng(seed); return {"tok": r.uniform(0, np.pi, (V, 3)), "pos": r.uniform(0, np.pi, (T, 3)), "q": r.uniform(0, np.pi, 6), "k": r.uniform(0, np.pi, 6)}
flat = lambda t: np.concatenate([t[k].ravel() for k in KEYS]); SH = [(V, 3), (T, 3), (6,), (6,)]
def unflat(x):
    out, o = {}, 0
    for k, sh in zip(KEYS, SH): n_ = int(np.prod(sh)); out[k] = x[o:o + n_].reshape(sh); o += n_
    return out
print(f"vocabulary {V}, block {T}, {len(tr_i)} training / {len(te_i)} held-out molecules; angles: {len(flat(init_theta(0)))} (3*{V} + 3*{T} + 12)")

In [ ]:
def init_c(mode, seed):
    r = np.random.default_rng(seed); n = lambda *s, sc=0.1: r.normal(0, sc, s); d = D
    p = {"E": n(V, d), "P": n(T, d, sc=0.05), "Wv": n(d, d, sc=d ** -.5), "Wo": n(d, d, sc=d ** -.5), "W1": n(d, 4 * d, sc=d ** -.5), "b1": np.zeros(4 * d), "W2": n(4 * d, d, sc=(4 * d) ** -.5), "b2": np.zeros(d), "Wout": n(d, V, sc=d ** -.5), "bout": np.zeros(V)}
    if mode == "classical": p["Wq"] = n(d, d, sc=d ** -.5); p["Wk"] = n(d, d, sc=d ** -.5)
    if mode == "rank3": p["Wq"] = n(d, 3, sc=d ** -.5); p["Wk"] = n(d, 3, sc=d ** -.5)
    return p
def fwd(p, ids, mode, S_ext=None):
    B_, T_ = ids.shape; X = p["E"][ids] + p["P"][:T_]; M = np.triu(np.ones((T_, T_), bool), 1); ca = {}
    if mode in ("classical", "rank3"): Q, K = X @ p["Wq"], X @ p["Wk"]; S = Q @ K.transpose(0, 2, 1) / math.sqrt(Q.shape[-1]); ca.update(Q=Q, K=K)
    elif mode == "hybrid": S = S_ext
    else: S = np.zeros((B_, T_, T_))
    A = softmax(np.where(M, -1e9, S)); Vv = X @ p["Wv"]; O = A @ Vv; X1 = X + O @ p["Wo"]; Hh = np.maximum(0, X1 @ p["W1"] + p["b1"]); X2 = X1 + Hh @ p["W2"] + p["b2"]
    ca.update(X=X, A=A, V=Vv, O=O, X1=X1, H=Hh, X2=X2, ids=ids); return X2 @ p["Wout"] + p["bout"], ca
def loss_grads(p, ids_in, targets, mode, S_ext=None, need_grad=True):
    logits, ca = fwd(p, ids_in, mode, S_ext); B_, T_, _ = logits.shape; mask = (targets != 0).astype(float); Pr = softmax(logits); idx = (np.arange(B_)[:, None], np.arange(T_)[None, :], targets)
    loss = -(np.log(Pr[idx] + 1e-12) * mask).sum() / mask.sum()
    if not need_grad: return loss, None, (Pr, mask)
    dl = Pr.copy(); dl[idx] -= 1; dl *= mask[:, :, None] / mask.sum(); g = {}
    g["Wout"] = np.einsum("btd,btv->dv", ca["X2"], dl); g["bout"] = dl.sum((0, 1)); dX2 = dl @ p["Wout"].T
    g["W2"] = np.einsum("bth,btd->hd", ca["H"], dX2); g["b2"] = dX2.sum((0, 1)); dH = (dX2 @ p["W2"].T) * (ca["H"] > 0); g["W1"] = np.einsum("btd,bth->dh", ca["X1"], dH); g["b1"] = dH.sum((0, 1)); dX1 = dX2 + dH @ p["W1"].T
    g["Wo"] = np.einsum("btd,bte->de", ca["O"], dX1); dO = dX1 @ p["Wo"].T; dA = dO @ ca["V"].transpose(0, 2, 1); dV = ca["A"].transpose(0, 2, 1) @ dO; g["Wv"] = np.einsum("btd,bte->de", ca["X"], dV); dX = dX1 + dV @ p["Wv"].T
    dS = ca["A"] * (dA - (dA * ca["A"]).sum(-1, keepdims=True))
    if mode in ("classical", "rank3"):
        sc = 1 / math.sqrt(ca["Q"].shape[-1]); dQ = dS @ ca["K"] * sc; dK = dS.transpose(0, 2, 1) @ ca["Q"] * sc; g["Wq"] = np.einsum("btd,bte->de", ca["X"], dQ); g["Wk"] = np.einsum("btd,bte->de", ca["X"], dK); dX = dX + dQ @ p["Wq"].T + dK @ p["Wk"].T
    g["E"] = np.zeros_like(p["E"]); np.add.at(g["E"], ca["ids"], dX); g["P"] = np.zeros_like(p["P"]); g["P"][:T_] = dX.sum(0); return loss, g, (Pr, mask)
# gradient check of the classical part (full classical model), a few random weights
pc = init_c("classical", 0); b_ = tr_i[:4]; inp_ = np.concatenate([np.ones((4, 1), int), ids_all[b_][:, :-1]], 1); l0, g0 = loss_grads(pc, inp_, ids_all[b_], "classical")[:2]; worst = 0.0; r_ = np.random.default_rng(3)
for k in pc:
    for _ in range(2):
        ix = tuple(r_.integers(0, s) for s in pc[k].shape); a_, b2_ = ({q: z_.copy() for q, z_ in pc.items()} for _ in range(2)); a_[k][ix] += 1e-6; b2_[k][ix] -= 1e-6
        worst = max(worst, abs((loss_grads(a_, inp_, ids_all[b_], "classical", need_grad=False)[0] - loss_grads(b2_, inp_, ids_all[b_], "classical", need_grad=False)[0]) / 2e-6 - g0[k][ix]))
print(f"gradient check of the classical part: largest difference {worst:.1e}"); assert worst < 1e-7

In [ ]:
def run(mode, steps=1200, bs=48, seed=1, lr=3e-3, lr_theta=2e-2, eps=0.01):
    p = init_c(mode, seed); rg = np.random.default_rng(seed); m_ = {k: np.zeros_like(v) for k, v in p.items()}; v_ = {k: np.zeros_like(x) for k, x in p.items()}; hist, val = [], []
    if mode == "hybrid": x = flat(init_theta(seed)); mt = np.zeros_like(x); vt = np.zeros_like(x)
    def val_loss():
        ev = te_i; inp = np.concatenate([np.ones((len(ev), 1), int), ids_all[ev][:, :-1]], 1); S = q_scores(unflat(x), inp) if mode == "hybrid" else None
        l, _, (Pr, mask) = loss_grads(p, inp, ids_all[ev], mode, S, False); return l, ((Pr.argmax(-1) == ids_all[ev]) * mask).sum() / mask.sum()
    for t in range(1, steps + 1):
        b = tr_i[rg.integers(0, len(tr_i), bs)]; inp = np.concatenate([np.ones((bs, 1), int), ids_all[b][:, :-1]], 1); tgt = ids_all[b]
        S = q_scores(unflat(x), inp) if mode == "hybrid" else None; l, g, _ = loss_grads(p, inp, tgt, mode, S); hist.append(l)
        for k in p: m_[k] = 0.9 * m_[k] + 0.1 * g[k]; v_[k] = 0.999 * v_[k] + 0.001 * g[k] ** 2; p[k] -= lr * (m_[k] / (1 - 0.9 ** t)) / (np.sqrt(v_[k] / (1 - 0.999 ** t)) + 1e-8)
        if mode == "hybrid":      # SPSA on all angles: two extra loss evaluations whatever the number of angles
            Dl = rg.choice([-1.0, 1.0], size=x.shape); lp = loss_grads(p, inp, tgt, mode, q_scores(unflat(x + eps * Dl), inp), False)[0]; lm = loss_grads(p, inp, tgt, mode, q_scores(unflat(x - eps * Dl), inp), False)[0]
            gt = (lp - lm) / (2 * eps) * Dl; mt = 0.9 * mt + 0.1 * gt; vt = 0.999 * vt + 0.001 * gt ** 2; x = x - lr_theta * (mt / (1 - 0.9 ** t)) / (np.sqrt(vt / (1 - 0.999 ** t)) + 1e-8)
        if t % 100 == 0: val.append((t, *val_loss()))
    return val, hist
N_SEEDS = 1; results = {}; t0 = time.perf_counter()
for mode in ("none", "rank3", "classical", "hybrid"):
    runs = [run(mode, seed=s + 1) for s in range(N_SEEDS)]; results[mode] = runs; print(f"{mode:10s} validation loss {np.mean([r[0][-1][1] for r in runs]):.4f}, next-token accuracy {np.mean([r[0][-1][2] for r in runs]):.3f} [{time.perf_counter() - t0:.0f} s]")

In [ ]:
names = {"none": "attention-free (scores = 0)", "rank3": "classical, rank-3 Q and K", "classical": "classical, 64x64 Q and K", "hybrid": "hybrid, SPSA on the angles"}
summ = pd.DataFrame([(names[m], np.mean([r[0][-1][1] for r in results[m]]), np.mean([r[0][-1][2] for r in results[m]])) for m in results], columns=["attention layer", "validation loss", "next-token accuracy"]); print(summ.round(4).to_string(index=False))
fig, ax = plt.subplots(figsize=(6.6, 3.8)); cols = dict(none=COLORS["gray"], rank3=COLORS["green"], classical=COLORS["blue"], hybrid=COLORS["orange"])
for m, runs in results.items(): ax.plot([v[0] for v in runs[0][0]], [np.mean([r[0][i][1] for r in runs]) for i in range(len(runs[0][0]))], color=cols[m], label=names[m])
ax.set(xlabel="training step", ylabel="validation cross-entropy", title="miniature decoder, four attention layers"); ax.legend(fontsize=8); plt.tight_layout(); plt.show()
hl, nl, cl = summ["validation loss"][3], summ["validation loss"][0], summ["validation loss"][2]
print(f"hybrid layer: validation loss {hl:.3f}; attention-free {nl:.3f}; classical {cl:.3f}. The hybrid layer is {'below' if hl < nl else 'not below'} the attention-free baseline and {hl - cl:.3f} above the classical layer in this run (the post averages three seeds); the paper reports parity after 20 epochs on QM9")
assert all(s < 0.6 for s in summ["validation loss"])

## 9. Your turn
**Exercise 1.** Change `N_SEEDS` to $3$ and compare the spread over seeds with the differences between the four layers.
**Exercise 2.** Give the hybrid model $4$ or $16$ SPSA directions per step (average the estimates; the cost is $8$ or $32$ evaluations) and see whether it approaches the classical layers.
**Exercise 3.** Add a second ansatz layer to $U_q$ and $U_k$ (twice the angles). Does the cancellation of the entangling layers of Section 4 still happen, and what is the rank of the Jacobian now?
**Exercise 4.** Estimate the scores with a finite number of shots in the forward pass of the training, $N=1{,}000$, and see what the shot noise does to the loss.
**Exercise 5.** Use the exact derivative $f'(\theta)=\tfrac12f(\theta+\pi)$ instead of SPSA for the $12$ angles of $U_q$ and $U_k$ only and compare.

## 10. Self-check

In [ ]:
assert max(errs) < 1e-12 and got["q_seq"] == 47704 and bad == [("sequence only", "Classical")] and abs(np.mean(np.abs(two_term[ok] / exact[ok])) - math.sqrt(2)) < 1e-3
print("All checks passed: you reproduced the numbers in the post.")

# Part II. CUDA-Q and the repository
> **Read this first.** Part II has three guarded stages. **(1)** CUDA-Q (the simulator used in the paper): the cell installs `cudaq` if it is missing, builds the ansatz of Section 2 as a CUDA-Q kernel (Ry gates and a ring of CNOTs) and compares its state vector with ours (up to the qubit-ordering convention). **(2)** The repository `anthonysmaldone/Quantum-Transformer`: the cell clones it, loads the model checkpoints in `model_checkpoints/` on the CPU and counts their parameters, which are compared with the six numbers of the paper ($47{,}704$, $55{,}713$, $48{,}307$, $56{,}535$ and the two of the parameter-matched classical models). **(3)** The models of the repository are instantiated with the vocabulary and block size of Section 7 and their parameter counts are compared with the paper's. The repository pins `cudaq==0.9.1`, PyTorch $2.5.1$ and RDKit; installing them in a new runtime may take a few minutes or fail, in which case the cell says why. The last cell lists the checks that passed.

In [ ]:
checks = []
def check(label, ok): checks.append((label, bool(ok))); print(("   ok   " if ok else "  FAIL  ") + label)
HAVE_CUDAQ = importlib.util.find_spec("cudaq") is not None
if not HAVE_CUDAQ:
    try:
        import subprocess; subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "cudaq"]); HAVE_CUDAQ = importlib.util.find_spec("cudaq") is not None
    except Exception as e_: print("cudaq could not be installed:", type(e_).__name__, str(e_)[:200])
if HAVE_CUDAQ:
    try:
        import cudaq; cudaq.set_target("qpp-cpu") if hasattr(cudaq, "set_target") else None
        @cudaq.kernel
        def ansatz6k(t: list[float]):
            q = cudaq.qvector(6)
            ry(t[0], q[0]); ry(t[1], q[1]); ry(t[2], q[2]); ry(t[3], q[3]); ry(t[4], q[4]); ry(t[5], q[5])
            x.ctrl(q[0], q[1]); x.ctrl(q[1], q[2]); x.ctrl(q[2], q[3]); x.ctrl(q[3], q[4]); x.ctrl(q[4], q[5]); x.ctrl(q[5], q[0])
        t6 = list(rng.uniform(0, np.pi, 6)); s_cq = np.array(cudaq.get_state(ansatz6k, t6)); z0 = np.zeros(64); z0[0] = 1; s_np = ansatz(np.array(t6)) @ z0
        rev = np.array([int(format(i, "06b")[::-1], 2) for i in range(64)])
        check("the CUDA-Q kernel (Ry layer + CNOT ring on 6 qubits) gives our state vector (either qubit ordering)", np.allclose(s_cq, s_np, atol=1e-6) or np.allclose(s_cq, s_np[rev], atol=1e-6))
    except Exception as e_: print("CUDA-Q stage could not be completed:", type(e_).__name__, str(e_)[:300])
else: print("CUDA-Q is not available: the CUDA-Q stage was skipped.")

In [ ]:
REPO = "Quantum-Transformer"; paper_counts = {47704, 55713, 48307, 56535}; ckpt_counts = {}
try:
    import subprocess
    if not os.path.isdir(REPO): subprocess.check_call(["git", "clone", "--depth", "1", "https://github.com/anthonysmaldone/Quantum-Transformer.git", REPO])
    print("repository files:", sorted(os.listdir(REPO))[:20])
    import torch
    for f in sorted(glob.glob(os.path.join(REPO, "model_checkpoints", "**", "*.pt"), recursive=True))[:12]:
        try:
            ck = torch.load(f, map_location="cpu", weights_only=False); sd = ck["model_state_dict"] if isinstance(ck, dict) and "model_state_dict" in ck else ck
            n_ = sum(v.numel() for v in sd.values() if hasattr(v, "numel")); ckpt_counts[f] = n_; print(f"{os.path.relpath(f, REPO)}: {n_:,} parameters; metrics: {ck.get('metrics') if isinstance(ck, dict) else None}")
        except Exception as e_: print("could not read", f, type(e_).__name__)
    if ckpt_counts: check("the parameter counts of the repository's checkpoints are among the six numbers of the paper", all(n_ in paper_counts or n_ in {47704, 48307} for n_ in ckpt_counts.values()))
    else: print("no checkpoints were found or readable")
except Exception as e_: print("repository stage could not be completed:", type(e_).__name__, str(e_)[:300])

In [ ]:
try:
    sys.path.insert(0, os.path.abspath(REPO)); from src.transformer import Transformer_Model
    cfg = dict(qpu_count=1, vocab_size=33, embed_dim=64, block_size=24); got_ = {}
    for name, kw in (("classical, sequence", dict(classical_attention=True, conditional_training=False)), ("classical-eq, sequence", dict(classical_attention=True, conditional_training=False, classical_parameter_reduction=True)),
                     ("classical, conditions", dict(classical_attention=True, conditional_training=True)), ("hybrid, sequence", dict(classical_attention=False, conditional_training=False)), ("hybrid, conditions", dict(classical_attention=False, conditional_training=True))):
        try: m = Transformer_Model(**cfg, **kw); got_[name] = sum(p.numel() for p in m.parameters()); print(f"{name:24s} {got_[name]:,}")
        except Exception as e_: print(f"{name:24s} could not be built:", type(e_).__name__, str(e_)[:150])
    for name, want in (("classical, sequence", 55713), ("hybrid, sequence", 47704), ("hybrid, conditions", 48307), ("classical, conditions", 56535)):
        if name in got_: check(f"the repository's {name} model has the {want:,} parameters of the paper (vocabulary 33, block 24)", got_[name] == want)
except Exception as e_: print("model stage could not be completed:", type(e_).__name__, str(e_)[:300])
if checks:
    ok = [lab for lab, p_ in checks if p_]; bad_ = [lab for lab, p_ in checks if not p_]; print(f"\nPart II: {len(ok)} of {len(checks)} checks passed")
    for lab in bad_: print("  not passed:", lab)

### What to look for
* **The ansatz.** If the CUDA-Q state agrees with ours up to a qubit reordering, the circuits of Sections 2--4 are what a simulator runs (for the ring-of-CNOTs reading of Figure 3).
* **The counts.** Parameter counts that match the paper confirm the architecture rebuilt in Section 7; a mismatch in the conditional classical model would show where the 182 parameters of the accounting come from. If the vocabulary or block size of the repository differ from $33$ and $24$, the counts of the cell for models built with them will differ and the checkpoints give the true numbers.
* **Not covered.** Training the hybrid model (CUDA-Q, a GPU, about $8$ hours per epoch on $4$ GPUs according to Table 1) is not attempted.